# 1 · Set Up the FABRIC Slice

Provision a 3-node slice (Alice / BMv2 switch / Bob), upload QFabric, install BMv2,
compile the P4 quantum-channel program, start the switch, and wire the data-plane network.

This notebook is a thin wrapper over the tested functions in `scripts/deploy_fabric.py`
(`create_slice`, `upload_project`, `install_deps`, `configure_switch`, `setup_dataplane_ips`),
so it stays in lock-step with the command-line deployer.

**Prerequisite:** `00_overview` (env check passed, FABRIC tokens configured).
After this notebook the slice is live and the data plane is running — go to `02_run_experiment`.

### At a glance
- **Purpose:** stand up the 3-node slice and bring the quantum data plane online.
- **Inputs:** `SLICE_NAME`, the three FABRIC sites, and `SCENARIO` (sets the initial loss threshold).
- **Outputs:** a live slice — BMv2 running on the switch with the fiber-loss + MAC-rewrite tables, and data-plane IPs (10.10.1.x) on Alice/Bob for the classical channel.
- **Runs on / runtime:** FABRIC JupyterHub; **~10–20 min** with the source build, or **~3–5 min** if you set `BMV2_IMAGE` (prebuilt container pull).
- **If something fails:** BMv2 build errors are almost always a missing apt dep on the switch — the error names it; add it to `scripts/install_bmv2.sh`. Re-running this notebook is safe (it reuses an existing slice and preserves the node venvs).

## Configuration

In [1]:
SLICE_NAME = 'qfabric-bb84-2'
SITE_ALICE = 'TACC'      # Alice site
SITE_BOB   = 'TACC'      # Bob site — keep all three on ONE site (distance is emulated); another site = WAN stress mode
SITE_SW    = 'TACC'      # BMv2 switch (colocated with Alice)
SCENARIO   = 'validation/scenarios/fabric_1km.yml'   # repo-relative; sizes the loss model

# Optional: run BMv2 from a PREBUILT container instead of compiling it on the
# switch (saves the multi-minute source build). Set to the published image
# 'ghcr.io/kthare10/qfabric-bmv2:latest' (or 'p4lang/p4c:latest'); leave '' to
# build BMv2 from source on the switch.
BMV2_IMAGE = 'ghcr.io/kthare10/qfabric-bmv2:latest'

In [2]:
import sys
from pathlib import Path

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

# deploy_fabric.py holds the tested provisioning/run logic; the notebooks are
# thin wrappers around it so there is a single source of truth.
import deploy_fabric as df
from qne.config import ScenarioConfig

fablib = df.get_fablib()

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/Users/kthare10/work/fabric_config/id_token.json
Project ID,4604cab7-41ff-4c1a-a935-0ca6f20cceeb
Bastion Host,bastion.fabric-testbed.net
Bastion Username,kthare10_0011904101
Bastion Private Key File,/Users/kthare10/.ssh/bastion-prod-2
Slice Public Key File,/Users/kthare10/.ssh/id_rsa.pub
Slice Private Key File,/Users/kthare10/.ssh/id_rsa


## Step 1 — Provision the slice (Alice, Bob, switch + L2 networks)
Submits the slice and waits for SSH. Takes a few minutes.

In [3]:
# Reuse the slice if it already exists (re-runnable); otherwise provision it.
try:
    slice_obj = fablib.get_slice(name=SLICE_NAME)
    print(f"Reusing existing slice '{SLICE_NAME}' (state: {slice_obj.get_state()})")
except Exception:
    slice_obj = df.create_slice(fablib, SLICE_NAME, SITE_ALICE, SITE_BOB, SITE_SW)


Retry: 8, Time: 194 sec


ID,d48de2ac-c825-444f-ba6f-3950d9738ef8
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-09-23 00:52:04 +0000
Lease Start (UTC),2026-09-22 00:52:04 +0000
Project ID,990d8a8b-7e50-4d13-a3be-0f133ffa8653
State,StableOK
Email,kthare10@email.unc.edu
UserId,43b7271b-90eb-45f6-833a-e51cf13bbc68


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
3e989a46-935f-4d34-aa57-4e02f3c3e652,alice,4,8,100,default_ubuntu_22,qcow2,tacc-w4.fabric-testbed.net,TACC,ubuntu,2605:2800:2011:201:f816:3eff:fe41:e772,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fe41:e772,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
ec2ae86c-013b-487d-b286-94a496249623,bob,4,8,100,default_ubuntu_22,qcow2,tacc-w4.fabric-testbed.net,TACC,ubuntu,2605:2800:2011:201:f816:3eff:feb8:6bb7,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2605:2800:2011:201:f816:3eff:feb8:6bb7,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
e49d006a-b2fc-4017-bb55-2a2b2dbaca20,switch,4,8,100,default_ubuntu_22,qcow2,tacc-w4.fabric-testbed.net,TACC,ubuntu,2605:2800:2011:201:f816:3eff:fe88:9d39,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fe88:9d39,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
5bf61b56-9a0c-4778-8e55-d7963651ea0c,net_alice_switch,L2,L2Bridge,TACC,None,None,Active,
2210afeb-7c02-41c9-88e7-4b16525e286a,net_switch_bob,L2,L2Bridge,TACC,None,None,Active,


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
alice-alice_nic-p1,p1,alice,net_alice_switch,100,,02:E2:4F:F1:D8:40,enp7s0,enp7s0,config,fe80::e2:4fff:fef1:d840,4,HundredGigE0/0/0/11
switch-sw_nic_bob-p1,p1,switch,net_switch_bob,100,,22:26:55:5E:E9:28,enp7s0,enp7s0,config,fe80::2026:55ff:fe5e:e928,4,HundredGigE0/0/0/11
switch-sw_nic_alice-p1,p1,switch,net_alice_switch,100,,26:0C:FA:6D:5F:8E,enp8s0,enp8s0,config,fe80::240c:faff:fe6d:5f8e,4,HundredGigE0/0/0/11
bob-bob_nic-p1,p1,bob,net_switch_bob,100,,1E:98:7C:47:C4:2D,enp7s0,enp7s0,config,fe80::1c98:7cff:fe47:c42d,4,HundredGigE0/0/0/11



Time to print interfaces 194 seconds
Waiting for slice to be ready...


Waiting for slice . Slice state: StableOK


Waiting for ssh in slice .

 ssh successful

=== Slice ready ===


ID,d48de2ac-c825-444f-ba6f-3950d9738ef8
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-09-23 00:52:04 +0000
Lease Start (UTC),2026-09-22 00:52:04 +0000
Project ID,990d8a8b-7e50-4d13-a3be-0f133ffa8653
State,StableOK
Email,kthare10@email.unc.edu
UserId,43b7271b-90eb-45f6-833a-e51cf13bbc68


## Step 2 — Upload QFabric and install dependencies
`upload_project` copies only the needed files (no `.venv`/`.git`). Then either:
- **source build** (`BMV2_IMAGE = ''`): `install_deps` sets up Alice/Bob Python deps and builds BMv2 on the switch (several minutes the first time); or
- **prebuilt image** (`BMV2_IMAGE` set): installs Alice/Bob deps, installs Docker on the switch and pulls the image, and enables the container path — no source build.

In [4]:
import os
df.upload_project(slice_obj)

if BMV2_IMAGE:
    os.environ['QFABRIC_BMV2_IMAGE'] = BMV2_IMAGE   # configure_switch uses the container
    df.install_deps(slice_obj, build_bmv2=False)    # Alice/Bob Python deps only
    df.setup_switch_docker(slice_obj, BMV2_IMAGE)   # install Docker + pull image on switch
else:
    df.install_deps(slice_obj)                      # build BMv2 from source on the switch


=== Uploading project (clean tarball) ===


  Uploading to alice...


  Uploading to bob...


  Uploading to switch...


  Upload complete (qne + validation + scenarios + p4 on every node)


...

5c7495f018c6: Pull complete


dd49370c9994: Pull complete


d85cbb7f9f3e: Pull complete
Digest: sha256:217abcecb0e1cca16bd84922a3982dacb9c117c7f36d03051b3b8e4f15fb46bb
Status: Downloaded newer image for ghcr.io/kthare10/qfabric-bmv2:latest
ghcr.io/kthare10/qfabric-bmv2:latest
--- Verifying the image has the BMv2 toolchain ---


1.15.3-unknown



=== Done. Enable the Docker path in the deployer with: ===
    export QFABRIC_BMV2_IMAGE='ghcr.io/kthare10/qfabric-bmv2:latest'


## Step 3 — Compile P4, start the switch, and set up the data plane
Computes the fiber-loss threshold from the scenario, starts BMv2 (durable `systemd-run`)
with the loss-model + MAC-rewrite tables, and assigns data-plane IPs so the classical
channel rides the FABRIC L2 link (the management network blocks cross-site TCP).

In [5]:
threshold = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO).loss_threshold_u32
print(f'P4 loss threshold (u32): {threshold}')

alice_mac, bob_mac, sw_alice_mac, sw_bob_mac, _, _ = df.configure_switch(slice_obj, threshold)
alice_ip, bob_ip = df.setup_dataplane_ips(slice_obj, alice_mac, bob_mac)
print('\nSlice ready: switch running, data plane up.')

P4 loss threshold (u32): 193305371

=== Configuring switch (threshold=193305371) ===


  Switch interfaces: enp8s0 (Alice), enp7s0 (Bob)


  Alice MAC: 02:E2:4F:F1:D8:40


  Bob MAC:   1E:98:7C:47:C4:2D


  Switch Alice-side MAC: 26:0C:FA:6D:5F:8E


...

  ARP: Alice → 10.10.1.2 via 26:0c:fa:6d:5f:8e (switch Alice-side)


  ARP: Bob → 10.10.1.1 via 22:26:55:5e:e9:28 (switch Bob-side)


  Testing connectivity (ping)...


  Ping successful!



Slice ready: switch running, data plane up.


---
**Next:** `02_run_experiment`.